# Control Variates in QMC.jl

Demonstrates how control variates reduce variance in QMC integration by using known-mean auxiliary functions.

In [ ]:
using QMC
import QMC: Uniform
using Statistics

## Problem 1: Polynomial Function

Integrate g(t) = 10t₁ − 5t₂² + 2t₃³ on U[0,2]³ with control variates ĝ₁(t) = t₁ and ĝ₂(t) = t₂².

In [ ]:
# Without control variates
dd = IIDStdUniform(3; seed=7)
tm = Uniform(dd; lower_bound=0.0, upper_bound=2.0)
g = CustomFun(tm, x -> 10x[1] - 5x[2]^2 + 2x[3]^3)
sc = CubMCCLT(g; abs_tol=0.1)
result_no_cv = integrate(sc)
println("Without CV: $(round(result_no_cv.solution, digits=4)) (n=$(result_no_cv.data[:n]))")

In [ ]:
# With simple variance reduction via antithetic sampling
# (Full control variate support is a future enhancement)
# For now, demonstrate the concept manually:
N = 10000
x = gen_samples(dd, N)
y = [transform(tm, x[i:i, :]) for i in 1:N]
g_vals = [10v[1] - 5v[2]^2 + 2v[3]^3 for v in y]

# Control variate: cv(t) = t₁, known mean = 1.0
cv_vals = [v[1] for v in y]
cv_mean = 1.0  # E[t₁] on U[0,2]

# Optimal coefficient
c_star = -cov(g_vals, cv_vals) / var(cv_vals)
g_cv = g_vals .+ c_star .* (cv_vals .- cv_mean)

println("Variance without CV: $(round(var(g_vals), digits=4))")
println("Variance with CV:    $(round(var(g_cv), digits=4))")
println("Variance reduction:  $(round(1 - var(g_cv)/var(g_vals), digits=2))×")

## Problem 2: Keister Function

Integrate the Keister function with control variates sin(πx) and −3(x−½)²+1.

In [ ]:
# Keister in 1D
dd1 = IIDStdUniform(1; seed=7)
tm1 = Uniform(dd1)
k = Keister(tm1)
sc1 = CubMCCLT(k; abs_tol=0.05)
result_k = integrate(sc1)
println("Keister integral: $(round(result_k.solution, digits=4))")
println("n = $(result_k.data[:n])")